## Environment & Imports

In [1]:
import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

# Initialize environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Environment Error: OPENAI_API_KEY is not defined.")
    
print("✅ Environment initialized.")

✅ Environment initialized.


## Define State & Nodes

In [2]:
# 1. Define the State Schema (TypedDict)
# 'add_messages' appends new messages to the existing list rather than overwriting
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# 3. Define the core node function
def chatbot_node(state: State):
    """Processes the current state messages and returns the LLM response."""
    response = llm.invoke(state["messages"])
    # Return a dictionary updating the 'messages' key
    return {"messages": [response]}

print("✅ State and Nodes defined.")

✅ State and Nodes defined.


## Compile and Execute the Graph

In [3]:
# 1. Build the StateGraph
graph_builder = StateGraph(State)

# 2. Add nodes and define routing edges
graph_builder.add_node("chatbot", chatbot_node)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# 3. Compile the graph into an executable Runnable
app = graph_builder.compile()

# 4. Execute the workflow
initial_state = {"messages": ["Explain the difference between LangChain and LangGraph in two sentences."]}
result = app.invoke(initial_state)

print("--- Execution Output ---\n")
print(result["messages"][-1].content)

--- Execution Output ---

LangChain is a framework designed for building applications that utilize language models, providing tools for managing prompts, chains, and agents to enhance interaction with these models. In contrast, LangGraph focuses on representing and manipulating knowledge graphs, enabling users to integrate and query structured data alongside language model capabilities for more complex applications.


## Automated State Validation

In [4]:
# Validate the execution output structure
print("--- Running Graph Validations ---")

try:
    # Check if the state dictionary contains the required keys
    assert "messages" in result, "Failure: State dictionary is missing the 'messages' key."
    
    # Check if the LLM actually appended a response to the initial prompt
    assert len(result["messages"]) > 1, "Failure: The graph did not append an AI response to the state."
    
    # Check if the final message contains actual text
    assert len(result["messages"][-1].content) > 0, "Failure: The AI returned an empty string."
    
    print("✅ All graph validations passed. The LangGraph architecture is fully functional.")
    
except AssertionError as e:
    print(f"❌ Test Failed: {e}")

--- Running Graph Validations ---
✅ All graph validations passed. The LangGraph architecture is fully functional.
